# Bronze-to-Silver validation audit for ImagingDicom

Use this notebook as a diagnostic gate between the Bronze ImagingDicom table and the Silver flattening step.
It scans the Bronze Delta table for malformed JSON, empty payloads, and unexpected resource structures before Silver processing runs.

> This notebook is scoped to /Tables/ImagingDicom only.


In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StringType


def validate_payload(payload):
    if payload is None or not isinstance(payload, str) or not payload.strip():
        return "EMPTY_OR_NULL_RESOURCE"

    try:
        parsed = json.loads(payload)
    except Exception as exc:
        return f"JSON_PARSE_ERROR: {type(exc).__name__}: {exc}"

    if not isinstance(parsed, dict):
        return "PAYLOAD_NOT_OBJECT"

    resource_type = parsed.get("resourceType")
    if not isinstance(resource_type, str) or not resource_type.strip():
        return "MISSING_RESOURCE_TYPE"

    return None


validate_udf = F.udf(validate_payload, StringType())

print("Validation helpers ready.")


: 

In [ ]:
# --- Configure these values before running the audit ---
BRONZE_TABLE_PATH = "abfss://ad5aafc2-395b-442a-9fd0-b1ba87aa067b@onelake.dfs.fabric.microsoft.com/5b4fecca-3520-4d8f-8c19-6f9780be19bd/Tables/ImagingDicom"
BAD_DATA_OUTPUT_PATH = "/lakehouse/default/Files/bronze_to_silver_validation/imagingdicom_invalid_rows"
SUMMARY_OUTPUT_PATH = "/lakehouse/default/Files/bronze_to_silver_validation/imagingdicom_summary"

RESOURCE_FILTER = None  # Example: ["ImagingStudy"]
MAX_SAMPLE_ROWS = 25
FAIL_ON_BAD_ROW_PCT = 0.01

print("Bronze table path:", BRONZE_TABLE_PATH)
print("Output path:", BAD_DATA_OUTPUT_PATH)
print("Summary output path:", SUMMARY_OUTPUT_PATH)
print("Resource filter:", RESOURCE_FILTER)


In [ ]:
try:
    bronze_df = spark.read.format("delta").load(BRONZE_TABLE_PATH)
except Exception as exc:
    raise RuntimeError(f"Unable to read Bronze Delta table from {BRONZE_TABLE_PATH}: {exc}") from exc

if RESOURCE_FILTER:
    bronze_df = bronze_df.filter(F.col("resourceType").isin(RESOURCE_FILTER))

bronze_df = bronze_df.cache()

print("Bronze row count:", bronze_df.count())
print("Bronze schema:")
bronze_df.printSchema()

sample_df = bronze_df.select("resourceType", "resource", "sourceSystem", "filePath").dropna(subset=["resource"]).limit(5)
display(sample_df)


In [ ]:
# Sample records for manual review before the Silver path writes anything.
preview_df = bronze_df.select("resourceType", "resource", "sourceSystem", "filePath").dropna(subset=["resource"]).limit(MAX_SAMPLE_ROWS)

print("Sample row count:", preview_df.count())
display(preview_df)


In [ ]:
# Infer a working schema from a clean sample of valid JSON and then validate every row with from_json.
clean_sample_df = bronze_df.filter(validate_udf(F.col("resource")).isNull()).select("resource").limit(1000)

if clean_sample_df.rdd.isEmpty():
    raise RuntimeError("No valid JSON resource rows were found in the Bronze source to infer a schema.")

sample_json_rdd = clean_sample_df.rdd.map(lambda row: row["resource"])
inferred_schema = spark.read.json(sample_json_rdd).schema

parsed_df = bronze_df.withColumn("parsed_resource", F.from_json(F.col("resource"), inferred_schema))

validation_df = (
    parsed_df
    .withColumn(
        "validation_error",
        F.when(F.col("parsed_resource").isNull(), "SCHEMA_PARSE_FAILED")
         .otherwise(validate_udf(F.col("resource")))
    )
    .withColumn("resource_preview", F.expr("substring(resource, 1, 180)"))
    .filter(F.col("validation_error").isNotNull())
)

print("Inferred schema:", inferred_schema)
print("Invalid payload row count:", validation_df.count())
display(validation_df.limit(100))


In [ ]:
total_rows = bronze_df.count()
invalid_rows = validation_df.count()
invalid_pct = invalid_rows / total_rows if total_rows else 0.0

summary_df = (
    validation_df.groupBy("resourceType", "validation_error")
    .agg(F.count("*").alias("invalid_rows"))
    .orderBy(F.desc("invalid_rows"))
)

print("Total Bronze rows:", total_rows)
print("Invalid payload rows:", invalid_rows)
print("Invalid payload rate:", f"{invalid_pct:.4%}")
display(summary_df)

if invalid_pct > FAIL_ON_BAD_ROW_PCT:
    print("WARNING: invalid Bronze payloads exceed the configured threshold.")
else:
    print("INFO: invalid Bronze payloads are within the configured threshold.")


In [ ]:
# Persist the diagnostic report and summary so the team can review the failures before rerunning Silver.
invalid_report_df = (
    validation_df.select(
        "resourceType",
        "validation_error",
        "resource_preview",
        "sourceSystem",
        "filePath",
        "resource"
    )
)

invalid_report_df.write.mode("overwrite").format("delta").save(BAD_DATA_OUTPUT_PATH)
summary_df.withColumn("invalid_pct", F.lit(invalid_pct)).write.mode("overwrite").format("delta").save(SUMMARY_OUTPUT_PATH)

print("Wrote invalid payload report to:", BAD_DATA_OUTPUT_PATH)
print("Wrote summary metrics to:", SUMMARY_OUTPUT_PATH)


In [ ]:
# Simple pass/fail gate for downstream Silver flattening.
if invalid_pct > FAIL_ON_BAD_ROW_PCT:
    raise RuntimeError(
        f"Bronze validation failed: {invalid_pct:.2%} invalid payloads exceed the {FAIL_ON_BAD_ROW_PCT:.2%} threshold."
    )

print("Bronze validation passed. The downstream Silver flattening step can proceed.")
